In [ ]:
####git? ?? ??####

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
)
import joblib
from module.dataload import load_data

In [ ]:
################################
# ???? ??? ?, ??? ??? ?? ?? ??? ???? ??
#################################

In [ ]:
# Load dataset
data = pd.read_csv("data/preprocessed.csv", index_col=0)
X_train, X_valid, y_train, y_valid, X_test, y_test = load_data(data, learning_type="ml")

In [ ]:
# Target ?? ??
data["label"].value_counts(normalize=True).plot(
    kind="bar", color=["skyblue", "lightcoral"]
)

In [ ]:
# DataLoad ??
# module/dataload? ???
# %%writefile module/ml_data.py

import pandas as pd


def load_dataset(file_path):
    data = pd.read_csv(file_path)
    data.columns = data.columns.str.lower()
    X = data.drop(["label"], axis=1)
    y = data["label"]

    return X, y


# dataset ??
from sklearn.model_selection import train_test_split

# X, y ??
X, y = load_dataset("data/preprocessed.csv")

# train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0
)
# train/valid
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=0
)

In [ ]:
print(X.head())
print(y.head())

In [ ]:
X_train, X_valid, y_train, y_valid, X_test, y_test = load_data(data, learning_type="ml")

In [ ]:
# Baseline model
models = [
    ("LR", LogisticRegression(random_state=0, max_iter=3000)),
    ("GBM", GradientBoostingClassifier(random_state=0)),
    ("RF", RandomForestClassifier(random_state=0)),
    ("XGB", XGBClassifier(random_state=0)),
]

In [ ]:
# ??

baseline_result_train_dict = {}
baseline_result_valid_dict = {}


for name, model in models:
    print(f"Training {name}...")
    model.fit(X_train, y_train)

    pred_valid = model.predict(X_valid)
    pred_proba_valid = model.predict_proba(X_valid)[:, 1]

    accuracy_valid = accuracy_score(y_valid, pred_valid)
    precision_valid = precision_score(y_valid, pred_valid)
    recall_valid = recall_score(y_valid, pred_valid)
    f1_valid = f1_score(y_valid, pred_valid)
    auc_valid = roc_auc_score(y_valid, pred_proba_valid)

    baseline_result_valid_dict[name] = [
        accuracy_valid,
        precision_valid,
        recall_valid,
        f1_valid,
        auc_valid,
    ]

In [ ]:
# result

pd.DataFrame(
    baseline_result_valid_dict,
    index=["accuracy", "precision", "recall", "f1", "roc-auc"],
)

In [ ]:
# rou-auc? ???????? ??? ?? ??? ??? ? ??
# rou-auc ???? LR? GBM? ??? ???? ?? ???? ??????? ?? ??

In [ ]:
######################
####tuning
#####################

In [ ]:
param_lr = {
    "C": [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
    "solver": ["liblinear", "saga"],
    "penalty": ["l1", "l2"],
    "max_iter": [100, 500, 1000],  # ????x
}

param_gbm = {
    "learning_rate": [0.001, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5],
    "n_estimators": [100, 500, 1000, 2000],
    "max_depth": range(1, 6),
    "subsample": [0.5, 0.6, 0.7, 0.8, 0.9, 1],
}

# Initialize RandomizedSearchCV for each model
random_search_lr = RandomizedSearchCV(
    LogisticRegression(random_state=0),
    param_lr,
    cv=4,
    scoring="roc_auc",
    n_jobs=-1,
    n_iter=60,
    random_state=0,
)
random_search_gbm = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=0),
    param_gbm,
    cv=4,
    scoring="roc_auc",
    n_jobs=-1,
    n_iter=60,
    random_state=0,
)

print("LogisticRegression ??...")
random_search_lr.fit(X_train, y_train)

print("GradientBoostingClassifier ??...")
random_search_gbm.fit(X_train, y_train)

In [ ]:
# tuning result
print("LogisticRegression Best parameters:", random_search_lr.best_params_)
print("GradientBoostingClassifier Best parameters:", random_search_gbm.best_params_)


print("LogisticRegression Best score:", random_search_lr.best_score_)
print("GradientBoostingClassifier Best score:", random_search_gbm.best_score_)

In [ ]:
# ?? ??

best_lr = random_search_lr.best_estimator_
best_gbm = random_search_gbm.best_estimator_

joblib.dump(best_lr, "model/best_lr.pkl")
joblib.dump(best_gbm, "model/best_gbm.pkl")

print("Models saved successfully.")

In [ ]:
###################
####??? ??####
###################

In [ ]:
# ??? ??? ??
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

# ?? ?? ???? ???
baseline_result_valid_dict = {}

for name, model in models:
    print(f"Training {name}...")
    model.fit(X_train, y_train)

    # ?? ?? ??
    pred_proba_valid = model.predict_proba(X_valid)[:, 1]

    # ? ???? ?? ?? ??
    for threshold in thresholds:
        # ??? ???? ?? ??
        pred_valid = (pred_proba_valid >= threshold).astype(int)

        # ?? ??
        accuracy_valid = accuracy_score(y_valid, pred_valid)
        precision_valid = precision_score(y_valid, pred_valid)
        recall_valid = recall_score(y_valid, pred_valid)
        f1_valid = f1_score(y_valid, pred_valid)
        auc_valid = roc_auc_score(
            y_valid, pred_proba_valid
        )  # ROC-AUC? ?? ???? ??

        # ?? ?? (? ?? ??? ???? ?? ??)
        baseline_result_valid_dict[(name, threshold)] = [
            accuracy_valid,
            precision_valid,
            recall_valid,
            f1_valid,
            auc_valid,
        ]

# ?? ??
import pandas as pd

results_df = pd.DataFrame(
    baseline_result_valid_dict,
    index=["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
).T
print(results_df)

In [ ]:
# roc-auc? ??? ??? ???? ??? ?? ??? ??

In [ ]:
##################################
###Feature importance ??####
##################################

In [ ]:
# LR feature importance
import matplotlib.pyplot as plt
import numpy as np

model_lr = LogisticRegression(random_state=0, max_iter=3000)

model_lr.fit(X_train, y_train)

# coefficient

coefficients = model_lr.coef_[0]
indices = np.argsort(np.abs(coefficients))
plt.figure(figsize=(10, 6))
plt.title("Feature Importance (LR)")
plt.barh(range(len(indices)), coefficients[indices], align="center")
plt.yticks(range(len(indices)), [X_train.columns[i] for i in indices])
plt.xlabel("Coefficient Value")
plt.show()

In [ ]:
# gbm feature importance
from sklearn.ensemble import GradientBoostingClassifier

# Gradient Boosting ?? ??
model_gbm = GradientBoostingClassifier(random_state=0)

model_gbm.fit(X_train, y_train)

# feature_importances_? ?? ?? ??? ???
importance = model_gbm.feature_importances_
indices = np.argsort(importance)
plt.figure(figsize=(10, 6))
plt.title("Feature Importance (Gradient Boosting Machine)")
plt.barh(range(len(indices)), importance[indices], align="center")
plt.yticks(range(len(indices)), [X_train.columns[i] for i in indices])
plt.xlabel("Importance")
plt.show()

In [ ]:
# GBM feature importance ???? ?????? ???? feature(device) ?? ? ??

In [ ]:
# DataLoad ??

# %%writefile module/ml_data.py

import pandas as pd


def load_dataset(file_path):
    data = pd.read_csv(file_path)
    data.columns = data.columns.str.lower()
    X = data.drop(["id", "label"], axis=1)
    y = data["label"]

    return X, y


# dataset ??
from sklearn.model_selection import train_test_split

# X, y ??
X, y = load_dataset("data/preprocessed_device_outlier.csv")

# train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0
)
# train/valid
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=0
)

In [ ]:
# Baseline model
models = [
    ("LR", LogisticRegression(random_state=0, max_iter=3000)),
    ("GBM", GradientBoostingClassifier(random_state=0)),
    ("RF", RandomForestClassifier(random_state=0)),
    ("XGB", XGBClassifier(random_state=0)),
]

In [ ]:
# ??

baseline_result_train_dict = {}
baseline_result_valid_dict = {}


for name, model in models:
    print(f"Training {name}...")
    model.fit(X_train, y_train)

    pred_valid = model.predict(X_valid)
    pred_proba_valid = model.predict_proba(X_valid)[:, 1]

    accuracy_valid = accuracy_score(y_valid, pred_valid)
    precision_valid = precision_score(y_valid, pred_valid)
    recall_valid = recall_score(y_valid, pred_valid)
    f1_valid = f1_score(y_valid, pred_valid)
    auc_valid = roc_auc_score(y_valid, pred_proba_valid)

    baseline_result_valid_dict[name] = [
        accuracy_valid,
        precision_valid,
        recall_valid,
        f1_valid,
        auc_valid,
    ]

In [ ]:
# result

pd.DataFrame(
    baseline_result_valid_dict,
    index=["accuracy", "precision", "recall", "f1", "roc-auc"],
)

In [ ]:
# device ?????? ???? ??? ??? ??
# ??????? ??

In [ ]:
param_lr = {
    "C": [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
    "solver": ["liblinear", "saga"],
    "penalty": ["l1", "l2"],
    "max_iter": [100, 500, 1000],  # ????x
}

test_results = {}

param_gbm = {
    "learning_rate": [0.001, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5],
    "n_estimators": [100, 500, 1000, 2000],
    "max_depth": range(1, 6),
    "subsample": [0.5, 0.6, 0.7, 0.8, 0.9, 1],
}

# Initialize RandomizedSearchCV for each model
random_search_lr = RandomizedSearchCV(
    LogisticRegression(random_state=0),
    param_lr,
    cv=4,
    scoring="roc_auc",
    n_jobs=-1,
    n_iter=60,
    random_state=0,
)
random_search_gbm = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=0),
    param_gbm,
    cv=4,
    scoring="roc_auc",
    n_jobs=-1,
    n_iter=60,
    random_state=0,
)

print("LogisticRegression ??...")
random_search_lr.fit(X_train, y_train)

print("GradientBoostingClassifier ??...")
random_search_gbm.fit(X_train, y_train)

In [ ]:
# tuning result
print("LogisticRegression Best parameters:", random_search_lr.best_params_)
print("GradientBoostingClassifier Best parameters:", random_search_gbm.best_params_)


print("LogisticRegression Best score:", random_search_lr.best_score_)
print("GradientBoostingClassifier Best score:", random_search_gbm.best_score_)

pd.DataFrame(test_results, index=["accuracy", "precision", "recall", "f1", "roc-auc"])

In [ ]:
# ?? ??

best_lr = random_search_lr.best_estimator_
best_gbm = random_search_gbm.best_estimator_

In [ ]:
# ?? ???
best_models = [
    ("LogisticRegression", best_lr),
    ("GradientBoostingClassifier", best_gbm),
]

test_results = {}

for name, model in best_models:
    print(f"?? {name}...")
    pred_test = model.predict(X_test)
    pred_proba_test = model.predict_proba(X_test)[:, 1]

    accuracy_test = accuracy_score(y_test, pred_test)
    precision_test = precision_score(y_test, pred_test)
    recall_test = recall_score(y_test, pred_test)
    f1_test = f1_score(y_test, pred_test)
    auc_test = roc_auc_score(y_test, pred_proba_test)

    test_results[name] = [accuracy_test, precision_test, recall_test, f1_test, auc_test]

In [ ]:
# ??
pd.DataFrame(test_results, index=["accuracy", "precision", "recall", "f1", "roc-auc"])

In [ ]:
# ?? ??

joblib.dump(best_lr, "model/best_lr_nodevice.pkl")
joblib.dump(best_gbm, "model/best_gbm_nodevice.pkl")

print("Models saved successfully.")

In [ ]:
# Confusion Matrix
conf_matrix = confusion_matrix(y_test, pred_test)
ConfusionMatrixDisplay(conf_matrix).plot()
plt.savefig("image/confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()